# Baseline: a Plain CNN Trained from Scratch for Lung Nodule Detection
Kaggle: enable **GPU** (Internet not needed). Same data, same 5 folds, same held-out test set and same metrics as the advanced pipeline, so the numbers are directly comparable.

**Goal:** understand exactly what a CNN does. No pretrained weights, no ready-made detector: just convolutions, a grid output, and a loss we write ourselves.

## 1. How a CNN works (what we build)
1. **Convolution:** a small filter (3×3 weights) slides over the image and outputs a high value where the local pattern matches the filter. The filter weights are what training learns.
2. **BatchNorm + ReLU:** normalize the values, then keep only positive responses (adds non-linearity).
3. **Max-pooling:** keep the strongest response in each 2×2 block. The image gets 2× smaller, and each pixel now "sees" a larger area.
4. **Repeat:** early layers detect edges and blobs, deeper layers detect combinations (a bright round spot inside dark lung).
5. **Detection head (grid):** after 4 poolings, 416×416 becomes a **26×26 grid**; each cell covers 16×16 pixels. For every cell the network predicts 5 numbers: `objectness` (is a nodule centered here?) and `x, y, w, h` (the box).
6. **Training:** the loss compares predictions with the true boxes, and gradients update all the filter weights.

## 2. Setup, data and folds

In [ ]:
# Install required evaluation packages
get_ipython().system("pip -q install torchmetrics pycocotools")

import gc
import glob
import os
import random

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from torchmetrics.detection import MeanAveragePrecision
from torchvision.ops import box_iou, nms

In [ ]:
# Global Configuration
SEED = 42
K_FOLDS = 5
EPOCHS = 60
BATCH_SIZE = 8
LEARNING_RATE = 1e-3

IMG_SIZE = 416
STRIDE = 16
GRID_SIZE = IMG_SIZE // STRIDE  # 26 x 26 grid
BOX_SCALE = 0.15 * IMG_SIZE     # box w, h predicted as a fraction of 62 px (nodules are ~8-35 px)


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# Locate dataset and build stratified 5-fold splits (same folds as the advanced pipeline)
DATA_ROOT = os.path.dirname(
    glob.glob("/kaggle/input/**/dataset_info.json", recursive=True)[0]
)


def load_split_items(split: str):
    img_paths = sorted(glob.glob(f"{DATA_ROOT}/{split}/images/*.jpg"))
    return [
        (p, f"{DATA_ROOT}/{split}/labels/{os.path.basename(p)[:-4]}.txt")
        for p in img_paths
    ]


pool_items = load_split_items("train") + load_split_items("val")
test_items = load_split_items("test")

strat_labels = [
    int(any(line.split()[0] == "1" for line in open(lbl_path).read().strip().splitlines()))
    for _, lbl_path in pool_items
]
skf = StratifiedKFold(K_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(pool_items)), strat_labels))

print(len(pool_items), "CV images,", len(test_items), "test images")

## 3. Dataset: image → grid target
Both classes are merged into one class "nodule" (as in the advanced pipeline). Each box is converted into a grid target: the cell containing the box center gets `objectness=1` and the box offsets inside that cell. Augmentation is minimal (flips + brightness).

In [ ]:
class NoduleGridDataset(torch.utils.data.Dataset):
    """Converts CT images and YOLO-format boxes into 26x26 grid detection targets."""

    def __init__(self, items, train: bool = False):
        self.items = items
        self.train = train

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        img_path, lbl_path = self.items[idx]
        img = TF.to_tensor(Image.open(img_path).convert("RGB"))

        boxes = []
        for line in open(lbl_path).read().strip().splitlines():
            _, x, y, w, h = map(float, line.split())
            boxes.append([
                (x - w / 2) * IMG_SIZE,
                (y - h / 2) * IMG_SIZE,
                (x + w / 2) * IMG_SIZE,
                (y + h / 2) * IMG_SIZE,
            ])
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)

        if self.train:
            if random.random() < 0.5:
                img = img.flip(-1)
                boxes[:, [0, 2]] = IMG_SIZE - boxes[:, [2, 0]]
            if random.random() < 0.5:
                img = img.flip(-2)
                boxes[:, [1, 3]] = IMG_SIZE - boxes[:, [3, 1]]
            img = (img * random.uniform(0.85, 1.15)).clamp(0, 1)

        # Target grid channels: [objectness, offset_x, offset_y, scaled_w, scaled_h]
        target = torch.zeros(5, GRID_SIZE, GRID_SIZE)
        for x1, y1, x2, y2 in boxes.tolist():
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            w = x2 - x1
            h = y2 - y1
            gj = min(int(cx // STRIDE), GRID_SIZE - 1)
            gi = min(int(cy // STRIDE), GRID_SIZE - 1)
            target[:, gi, gj] = torch.tensor([
                1,
                cx / STRIDE - gj,
                cy / STRIDE - gi,
                min(w / BOX_SCALE, 1),
                min(h / BOX_SCALE, 1),
            ])

        # Normalize pixels to [-1, 1]
        return (img - 0.5) / 0.5, target, boxes


def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    targets = torch.stack([item[1] for item in batch])
    gt_boxes = [item[2] for item in batch]
    return images, targets, gt_boxes


def build_dataloader(items, train: bool = False):
    return torch.utils.data.DataLoader(
        NoduleGridDataset(items, train=train),
        batch_size=BATCH_SIZE,
        shuffle=train,
        collate_fn=collate_fn,
    )

## 4. The CNN (4 conv blocks + 1 conv + 1×1 head, ~0.2 M parameters)

In [ ]:
def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 16),      # 416 -> 208
            conv_block(16, 32),     # 208 -> 104
            conv_block(32, 64),     # 104 -> 52
            conv_block(64, 128),    # 52  -> 26
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        # 5 numbers per grid cell: obj, x, y, w, h
        self.head = nn.Conv2d(128, 5, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


sample_model = BaselineCNN()
print(sum(p.numel() for p in sample_model.parameters()) / 1e6, "M parameters")
print(sample_model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape)

## 5. Loss, decoding, metrics
- **Loss** = BCE on objectness for all 676 cells (nodule cells are rare, so they get a larger weight) + L1 on the box numbers, only in cells that contain a nodule.
- **Decoding:** objectness → confidence score, offsets → box in pixels; then **NMS** removes duplicate boxes.
- **Metrics:** mAP@0.5 (threshold-free), and precision / recall / F1 at a confidence threshold tuned on out-of-fold predictions (never on test).

In [ ]:
POS_WEIGHT = 20.0


def loss_fn(out: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
    obj_loss = F.binary_cross_entropy_with_logits(
        out[:, 0],
        tgt[:, 0],
        pos_weight=torch.tensor(POS_WEIGHT, device=out.device),
    )
    pos_mask = tgt[:, 0] > 0
    if pos_mask.any():
        pred_boxes = torch.sigmoid(out[:, 1:]).permute(0, 2, 3, 1)[pos_mask]
        true_boxes = tgt[:, 1:].permute(0, 2, 3, 1)[pos_mask]
        box_loss = F.l1_loss(pred_boxes, true_boxes)
    else:
        box_loss = out.sum() * 0
    return obj_loss + 5 * box_loss


@torch.no_grad()
def decode_predictions(out: torch.Tensor, thr: float = 0.01):
    obj = torch.sigmoid(out[:, 0])
    reg = torch.sigmoid(out[:, 1:])
    results = []

    for k in range(out.shape[0]):
        gi, gj = (obj[k] > thr).nonzero(as_tuple=True)
        scores = obj[k][gi, gj]

        cx = (gj + reg[k, 0][gi, gj]) * STRIDE
        cy = (gi + reg[k, 1][gi, gj]) * STRIDE
        w = reg[k, 2][gi, gj] * BOX_SCALE
        h = reg[k, 3][gi, gj] * BOX_SCALE

        boxes = torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=1).clamp(0, IMG_SIZE)
        if len(boxes):
            keep = nms(boxes, scores, 0.5)[:30]
        else:
            keep = torch.zeros(0, dtype=torch.long, device=boxes.device)

        results.append({
            "boxes": boxes[keep].cpu(),
            "scores": scores[keep].cpu(),
            "labels": torch.ones(len(keep), dtype=torch.long),
        })
    return results


@torch.no_grad()
def predict_dataset(model: nn.Module, items):
    model.eval()
    outputs = []
    for x, _, gt_boxes in build_dataloader(items, train=False):
        preds = decode_predictions(model(x.to(DEVICE)))
        for pred, gt in zip(preds, gt_boxes):
            target = {
                "boxes": gt,
                "labels": torch.ones(len(gt), dtype=torch.long),
            }
            outputs.append((pred, target))
    return outputs


def compute_map(results):
    metric = MeanAveragePrecision(iou_type="bbox")
    metric.update([r[0] for r in results], [r[1] for r in results])
    computed = metric.compute()
    return float(computed["map_50"]), float(computed["map"])


def compute_prf(results, thr: float):
    tp = fp = fn = 0
    for pred, target in results:
        keep = pred["scores"] >= thr
        used_gt = set()
        for box in pred["boxes"][keep]:
            if len(target["boxes"]):
                ious = box_iou(box[None], target["boxes"])[0]
                best_j = int(ious.argmax())
                if ious[best_j] >= 0.5 and best_j not in used_gt:
                    tp += 1
                    used_gt.add(best_j)
                    continue
            fp += 1
        fn += len(target["boxes"]) - len(used_gt)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return precision, recall, f1


def find_best_threshold(results) -> float:
    thresholds = np.arange(0.02, 0.95, 0.02)
    return float(max(thresholds, key=lambda th: compute_prf(results, th)[2]))

## 6. Training with 5-fold cross-validation (fixed epochs, last-epoch weights, no checkpoint picking)

In [ ]:
oof_preds = []
test_preds = []
loss_curves = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    seed_all(SEED + fold_idx)
    model = BaselineCNN().to(DEVICE)
    train_loader = build_dataloader([pool_items[i] for i in train_idx], train=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    fold_curve = []

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for x, target, _ in train_loader:
            loss = loss_fn(model(x.to(DEVICE)), target.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        fold_curve.append(total_loss / len(train_loader))

    loss_curves.append(fold_curve)
    torch.save(model.state_dict(), f"baseline_cnn_f{fold_idx}.pt")

    val_res = predict_dataset(model, [pool_items[i] for i in val_idx])
    test_res = predict_dataset(model, test_items)
    oof_preds.append(val_res)
    test_preds.append(test_res)

    print(
        f"fold{fold_idx}: final loss {fold_curve[-1]:.3f} "
        f"| val mAP50 {compute_map(val_res)[0]:.3f} "
        f"| test mAP50 {compute_map(test_res)[0]:.3f}"
    )

    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

for curve in loss_curves:
    plt.plot(curve)
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.title("Training loss per fold")
plt.show()

## 7. Results (mean ± std over folds; threshold tuned on pooled out-of-fold predictions, applied to test)

In [ ]:
pooled_oof = [res for fold_res in oof_preds for res in fold_res]
best_thr = find_best_threshold(pooled_oof)
pooled_map50, pooled_map = compute_map(pooled_oof)
pooled_p, pooled_r, pooled_f1 = compute_prf(pooled_oof, best_thr)

print(f"Tuned threshold: {best_thr:.2f}")

summary_rows = {
    "CV (OOF pooled)": {
        "mAP50": pooled_map50,
        "mAP": pooled_map,
        "P": pooled_p,
        "R": pooled_r,
        "F1": pooled_f1,
    }
}

for label, fold_sets in [("CV folds (mean)", oof_preds), ("TEST per fold model (mean)", test_preds)]:
    fold_metrics = pd.DataFrame([
        {
            "mAP50": compute_map(r)[0],
            "mAP": compute_map(r)[1],
            **dict(zip(["P", "R", "F1"], compute_prf(r, best_thr))),
        }
        for r in fold_sets
    ])
    summary_rows[label] = fold_metrics.mean().to_dict()
    summary_rows[label.replace("(mean)", "(std)")] = fold_metrics.std().to_dict()

results_df = pd.DataFrame(summary_rows).T.round(3)
print(results_df)
results_df.to_csv("baseline_cnn_results.csv")

## 8. Look inside the CNN (fold 0)
- **Filters:** the 16 learned 3×3 filters of the first layer (these are the weights).
- **Feature maps:** what the first layer outputs for a CT slice.
- **Objectness heatmap:** the 26×26 grid of confidence, next to the ground truth (green) and the final detections (red).

In [ ]:
# Visualize Layer-1 learned 3x3 filters
model = BaselineCNN().to(DEVICE)
model.load_state_dict(torch.load("baseline_cnn_f0.pt"))
model.eval()

weights_l1 = model.features[0][0].weight.detach().cpu()
weights_l1 = (weights_l1 - weights_l1.min()) / (weights_l1.max() - weights_l1.min())

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, w in zip(axes.ravel(), weights_l1):
    ax.imshow(w.permute(1, 2, 0))
    ax.axis("off")
plt.suptitle("Layer-1 learned filters (3x3 RGB)")
plt.show()

# Visualize feature maps, objectness heatmap, and detections on 3 test slices
test_ds = NoduleGridDataset(test_items)
sample_indices = [i for i in range(len(test_ds)) if len(test_ds[i][2])][:3]

for idx in sample_indices:
    x, _, gt_boxes = test_ds[idx]
    with torch.no_grad():
        feat_maps = model.features[0][:3](x[None].to(DEVICE))[0].cpu()
        out = model(x[None].to(DEVICE))
        heatmap = torch.sigmoid(out[0, 0]).cpu()
        detections = decode_predictions(out, best_thr)[0]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    img_vis = (x * 0.5 + 0.5).permute(1, 2, 0).numpy()

    axes[0].imshow(img_vis)
    axes[0].set_title("Input + GT (green) + detections (red)")
    for b in gt_boxes:
        axes[0].add_patch(
            patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=2)
        )
    for b, score in zip(detections["boxes"], detections["scores"]):
        axes[0].add_patch(
            patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="red", lw=2)
        )
        axes[0].text(b[0], b[1] - 3, f"{score:.2f}", color="red")

    axes[1].imshow(feat_maps[0], cmap="gray")
    axes[1].set_title("Layer-1 feature map (channel 0)")

    axes[2].imshow(feat_maps[1], cmap="gray")
    axes[2].set_title("Layer-1 feature map (channel 1)")

    axes[3].imshow(heatmap, cmap="hot")
    axes[3].set_title("Objectness heatmap (26x26)")

    for ax in axes:
        ax.axis("off")
    plt.show()